In [5]:
import altair as alt
import numpy as np
import polars as pl

n = 2
T = 0.6
E = np.linspace(-4, 6, 201)


def e_squared(e):
    return e**n


def shifted_squared(e):
    return (e - 2) ** n


def both_squared(e):
    return e**n + (e - 2) ** n


def boltzmann(e):
    return np.exp(-e / T)


def weighted_boltzmann(e):
    return shifted_squared(e) * boltzmann(e)


weight1_scale = 200
weight2_scale = 200

data = {
    "E": E,
    "E^2": np.piecewise(E, [(E >= 0) & (E <= 4)], [e_squared, np.nan]),
    "(E-2)^2": np.piecewise(E, [(E >= 2) & (E <= 6)], [shifted_squared, np.nan]),
    "sum": np.piecewise(
        E,
        [(E >= 0) & (E < 2), (E >= 2) & (E <= 4)],
        [e_squared, both_squared, np.nan],
    ),
    "exp(-E/T)": np.piecewise(E, [(E >= 2) & (E <= 6)], [boltzmann, np.nan])
    * weight1_scale,
    "E^2 exp(-E/T)": np.piecewise(E, [(E >= 2) & (E <= 6)], [weighted_boltzmann, np.nan])
    * weight2_scale,
}
curves = [k for k in data if k != "E"]

df = pl.DataFrame(data).with_columns(pl.exclude("E").fill_nan(None))

chart = (
    alt.Chart(df)
    .transform_fold(curves, as_=["curve", "rho"])
    .transform_filter("isValid(datum.rho)")
    .mark_line()
    .encode(
        x=alt.X("rho:Q", title="ρ"),
        y=alt.Y("E:Q", title="E"),
        color=alt.Color("curve:N", title="Function", sort=curves),
        order=alt.Order("E:Q"),
    )
    .properties(width=400, height=400, title="ρ(E)")
)

chart

alt.Chart(...)